In [1]:
!pip install nltk wordcloud matplotlib seaborn scikit-learn tensorflow pandas numpy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re
import string
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_curve, auc, roc_auc_score)
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense, Dropout,
                                      Bidirectional, GlobalMaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping

print('✅ All libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


ValueError: JAX requires ml_dtypes version 0.5 or newer; installed version is 0.3.2.

In [ ]:
print('🔄 Performing a deep cleanup and reinstall of tensorflow and tensorflow-datasets...')
!pip uninstall tensorflow tensorflow-datasets protobuf tensorflow-metadata grpcio grpcio-status -y
!pip install tensorflow tensorflow-datasets -q

After running the above cell, please rerun the data loading cell (`pJjnvGZWQZtW`).

In [ ]:
import tensorflow_datasets as tfds

dataset, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)
train_data, test_data = dataset['train'], dataset['test']

train_texts, train_labels = [], []
for text, label in tfds.as_numpy(train_data):
    train_texts.append(text.decode('utf-8'))
    train_labels.append(label)

test_texts, test_labels = [], []
for text, label in tfds.as_numpy(test_data):
    test_texts.append(text.decode('utf-8'))
    test_labels.append(label)

df_train = pd.DataFrame({'review': train_texts, 'sentiment': train_labels})
df_test  = pd.DataFrame({'review': test_texts,  'sentiment': test_labels})
df = pd.concat([df_train, df_test], ignore_index=True)
df['sentiment_label'] = df['sentiment'].map({0: 'negative', 1: 'positive'})

print(f'Total samples: {len(df)}')
print(f'Train samples: {len(df_train)}')
print(f'Test samples : {len(df_test)}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('IMDB Sentiment Distribution', fontsize=15, fontweight='bold')

colors = ['#FF6B6B', '#4ECDC4']
counts = df['sentiment_label'].value_counts()

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Count of Reviews by Sentiment')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Sentiment Proportion')

plt.tight_layout()
plt.show()
print('
📊 Class Distribution:')
print(counts)

In [ ]:
df['review_length'] = df['review'].apply(lambda x: len(x.split()))
df['char_count']    = df['review'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Review Length Analysis', fontsize=14, fontweight='bold')

for sentiment, color in zip(['positive', 'negative'], colors):
    subset = df[df['sentiment_label'] == sentiment]['review_length']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=sentiment)
axes[0].set_title('Word Count Distribution by Sentiment')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].axvline(df['review_length'].mean(), color='black', linestyle='--', label='Mean')

df.boxplot(column='review_length', by='sentiment_label', ax=axes[1],
           boxprops=dict(color='navy'), medianprops=dict(color='red'))
axes[1].set_title('Word Count Box Plot by Sentiment')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word Count')
plt.suptitle('')

plt.tight_layout()
plt.show()

print('
📏 Review Length Stats:')
print(df.groupby('sentiment_label')['review_length'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Word Clouds: Most Frequent Words', fontsize=14, fontweight='bold')

stop_words = set(stopwords.words('english'))

for ax, sentiment, cmap in zip(axes, ['positive', 'negative'], ['Greens', 'Reds']):
    text = ' '.join(df[df['sentiment_label'] == sentiment]['review'].values)
    wc = WordCloud(width=700, height=400, background_color='white',
                   stopwords=stop_words, colormap=cmap,
                   max_words=100, collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{sentiment.upper()} Reviews', fontsize=13)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
lemmatizer = WordNetLemmatizer()
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text, use_lemma=True):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    if use_lemma:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    else:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

sample = df['review'][0]
print('📝 Original Review (first 300 chars):')
print(sample[:300])
print('\n✅ Preprocessed:')
print(preprocess_text(sample)[:300])

In [ ]:
print('⏳ Preprocessing text... (may take ~2 min)')
df['clean_review'] = df['review'].apply(preprocess_text)
print(f'✅ Done! Preprocessed {len(df)} reviews.')
df[['review', 'clean_review', 'sentiment_label']].head(3)

In [ ]:
X = df['clean_review'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {len(X_train)} samples')
print(f'Test size  : {len(X_test)}  samples')
print(f'Train class balance: {np.bincount(y_train)}')
print(f'Test  class balance: {np.bincount(y_test)}')

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

cv = CountVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
X_train_cv = cv.fit_transform(X_train)
X_test_cv  = cv.transform(X_test)

print(f'\nTF-IDF feature matrix shape: {X_train_tfidf.shape}')

In [ ]:
lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr  = lr_model.predict(X_test_tfidf)
y_prob_lr  = lr_model.predict_proba(X_test_tfidf)[:, 1]

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

print('=' * 50)
print('       MODEL 1: LOGISTIC REGRESSION')
print('=' * 50)
print(f'Accuracy : {lr_acc:.4f} ({lr_acc*100:.2f}%)')
print(f'ROC-AUC  : {lr_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression - Confusion Matrix', ax)
plt.tight_layout()
plt.show()

In [ ]:
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_cv, y_train)

y_pred_nb = nb_model.predict(X_test_cv)
y_prob_nb = nb_model.predict_proba(X_test_cv)[:, 1]

nb_acc = accuracy_score(y_test, y_pred_nb)
nb_auc = roc_auc_score(y_test, y_prob_nb)

print('=' * 50)
print('         MODEL 2: NAIVE BAYES')
print('=' * 50)
print(f'Accuracy : {nb_acc:.4f} ({nb_acc*100:.2f}%)')
print(f'ROC-AUC  : {nb_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_nb, target_names=['Negative', 'Positive']))

fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(y_test, y_pred_nb, 'Naive Bayes - Confusion Matrix', ax)
plt.tight_layout()
plt.show()

In [ ]:
MAX_VOCAB = 20000
MAX_LEN   = 200
EMBED_DIM = 64

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Vocab size    : {len(tokenizer.word_index)}')
print(f'Train pad shape: {X_train_pad.shape}')
print(f'Test  pad shape: {X_test_pad.shape}')

In [ ]:
def build_lstm_model(vocab_size, embed_dim, max_len):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        Bidirectional(LSTM(64, return_sequences=True)),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

lstm_model = build_lstm_model(MAX_VOCAB, EMBED_DIM, MAX_LEN)
lstm_model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = lstm_model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', marker='o', color='red')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='s', color='orange')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
y_prob_lstm = lstm_model.predict(X_test_pad).flatten()
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

lstm_acc = accuracy_score(y_test, y_pred_lstm)
lstm_auc = roc_auc_score(y_test, y_prob_lstm)

print('=' * 50)
print('      MODEL 3: BIDIRECTIONAL LSTM')
print('=' * 50)
print(f'Accuracy : {lstm_acc:.4f} ({lstm_acc*100:.2f}%)')
print(f'ROC-AUC  : {lstm_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lstm, target_names=['Negative', 'Positive']))

fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(y_test, y_pred_lstm, 'BiLSTM - Confusion Matrix', ax)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

models_results = {
    'Model': ['Logistic Regression', 'Naive Bayes', 'BiLSTM'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred_lr)*100, 2),
        round(accuracy_score(y_test, y_pred_nb)*100, 2),
        round(accuracy_score(y_test, y_pred_lstm)*100, 2)
    ],
    'Precision': [
        round(precision_score(y_test, y_pred_lr)*100, 2),
        round(precision_score(y_test, y_pred_nb)*100, 2),
        round(precision_score(y_test, y_pred_lstm)*100, 2)
    ],
    'Recall': [
        round(recall_score(y_test, y_pred_lr)*100, 2),
        round(recall_score(y_test, y_pred_nb)*100, 2),
        round(recall_score(y_test, y_pred_lstm)*100, 2)
    ],
    'F1-Score': [
        round(f1_score(y_test, y_pred_lr)*100, 2),
        round(f1_score(y_test, y_pred_nb)*100, 2),
        round(f1_score(y_test, y_pred_lstm)*100, 2)
    ],
    'ROC-AUC': [
        round(lr_auc*100, 2),
        round(nb_auc*100, 2),
        round(lstm_auc*100, 2)
    ]
}

results_df = pd.DataFrame(models_results)
results_df = results_df.set_index('Model')
print('\n🏆 MODEL COMPARISON SUMMARY (%)')
print('=' * 65)
print(results_df.to_string())
print('\n⭐ Best Model by Accuracy:', results_df['Accuracy'].idxmax())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df.index))
width = 0.15
metrics = results_df.columns.tolist()
bar_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

for i, (metric, color) in enumerate(zip(metrics, bar_colors)):
    offset = (i - len(metrics)//2) * width
    bars = ax.bar(x + offset, results_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xlabel('Models', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, fontsize=11)
ax.legend(loc='lower right')
ax.set_ylim(60, 102)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for (name, y_prob, color) in [
    ('Logistic Regression', y_prob_lr, '#3498db'),
    ('Naive Bayes',         y_prob_nb, '#e74c3c'),
    ('BiLSTM',             y_prob_lstm, '#2ecc71')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = lr_model.coef_[0]

top_pos_idx = np.argsort(coefs)[-20:]
top_neg_idx = np.argsort(coefs)[:20]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Top Predictive Words (Logistic Regression Coefficients)', fontsize=14, fontweight='bold')

axes[0].barh(feature_names[top_pos_idx], coefs[top_pos_idx], color='#2ecc71')
axes[0].set_title('Top 20 POSITIVE Words', fontweight='bold')
axes[0].set_xlabel('Coefficient Value')

axes[1].barh(feature_names[top_neg_idx], coefs[top_neg_idx], color='#e74c3c')
axes[1].set_title('Top 20 NEGATIVE Words', fontweight='bold')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

In [ ]:
def predict_sentiment(text, model_type='lr'):
    clean = preprocess_text(text)

    if model_type == 'lr':
        vec   = tfidf.transform([clean])
        prob  = lr_model.predict_proba(vec)[0][1]
        label = 'POSITIVE ✅' if prob >= 0.5 else 'NEGATIVE ❌'

    elif model_type == 'nb':
        vec   = cv.transform([clean])
        prob  = nb_model.predict_proba(vec)[0][1]
        label = 'POSITIVE ✅' if prob >= 0.5 else 'NEGATIVE ❌'

    elif model_type == 'lstm':
        seq   = tokenizer.texts_to_sequences([clean])
        pad   = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
        prob  = lstm_model.predict(pad, verbose=0)[0][0]
        label = 'POSITIVE ✅' if prob >= 0.5 else 'NEGATIVE ❌'

    confidence = prob if prob >= 0.5 else 1 - prob
    print(f'Sentiment  : {label}')
    print(f'Confidence : {confidence*100:.1f}%')
    print(f'Positive Prob: {prob*100:.1f}%')
    return label, confidence

test_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout.",
    "Terrible waste of time. The story made no sense and the characters were boring and one-dimensional.",
    "It was okay, not the best film but had some good moments. The special effects were decent."
]

for i, review in enumerate(test_reviews, 1):
    print(f'\n--- Review {i} ---')
    print(f'Text: {review[:80]}...' if len(review)>80 else f'Text: {review}')
    predict_sentiment(review, model_type='lr')

In [ ]:
print('=' * 55)
print('🎬 INTERACTIVE SENTIMENT ANALYZER')
print('=' * 55)
user_review = input('Enter a movie review: ')

print('\n📊 Predictions from all 3 models:')
for name, mtype in [('Logistic Regression', 'lr'), ('Naive Bayes', 'nb'), ('BiLSTM', 'lstm')]:
    print(f'\n🔹 {name}:')
    predict_sentiment(user_review, model_type=mtype)

In [ ]:
lemmatizer = WordNetLemmatizer()
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text, use_lemma=True):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    if use_lemma:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    else:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

sample = df['review'][0]
print('📝 Original Review (first 300 chars):')
print(sample[:300])
print('\n✅ Preprocessed:')
print(preprocess_text(sample)[:300])

In [ ]:
print('⏳ Preprocessing text... (may take ~2 min)')
df['clean_review'] = df['review'].apply(preprocess_text)
print(f'✅ Done! Preprocessed {len(df)} reviews.')
df[['review', 'clean_review', 'sentiment_label']].head(3)

In [ ]:
X = df['clean_review'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {len(X_train)} samples')
print(f'Test size  : {len(X_test)}  samples')
print(f'Train class balance: {np.bincount(y_train)}')
print(f'Test  class balance: {np.bincount(y_test)}')

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

cv = CountVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
X_train_cv = cv.fit_transform(X_train)
X_test_cv  = cv.transform(X_test)

print(f'\nTF-IDF feature matrix shape: {X_train_tfidf.shape}')

In [ ]:
lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr  = lr_model.predict(X_test_tfidf)
y_prob_lr  = lr_model.predict_proba(X_test_tfidf)[:, 1]

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

print('=' * 50)
print('       MODEL 1: LOGISTIC REGRESSION')
print('=' * 50)
print(f'Accuracy : {lr_acc:.4f} ({lr_acc*100:.2f}%)')
print(f'ROC-AUC  : {lr_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression - Confusion Matrix', ax)
plt.tight_layout()
plt.show()

In [ ]:
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_cv, y_train)

y_pred_nb = nb_model.predict(X_test_cv)
y_prob_nb = nb_model.predict_proba(X_test_cv)[:, 1]

nb_acc = accuracy_score(y_test, y_pred_nb)
nb_auc = roc_auc_score(y_test, y_prob_nb)

print('=' * 50)
print('         MODEL 2: NAIVE BAYES')
print('=' * 50)
print(f'Accuracy : {nb_acc:.4f} ({nb_acc*100:.2f}%)')
print(f'ROC-AUC  : {nb_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_nb, target_names=['Negative', 'Positive']))

fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(y_test, y_pred_nb, 'Naive Bayes - Confusion Matrix', ax)
plt.tight_layout()
plt.show()

In [ ]:
MAX_VOCAB = 20000
MAX_LEN   = 200
EMBED_DIM = 64

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Vocab size    : {len(tokenizer.word_index)}')
print(f'Train pad shape: {X_train_pad.shape}')
print(f'Test  pad shape: {X_test_pad.shape}')

In [ ]:
def build_lstm_model(vocab_size, embed_dim, max_len):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        Bidirectional(LSTM(64, return_sequences=True)),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

lstm_model = build_lstm_model(MAX_VOCAB, EMBED_DIM, MAX_LEN)
lstm_model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = lstm_model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', marker='o', color='red')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='s', color='orange')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
y_prob_lstm = lstm_model.predict(X_test_pad).flatten()
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

lstm_acc = accuracy_score(y_test, y_pred_lstm)
lstm_auc = roc_auc_score(y_test, y_prob_lstm)

print('=' * 50)
print('      MODEL 3: BIDIRECTIONAL LSTM')
print('=' * 50)
print(f'Accuracy : {lstm_acc:.4f} ({lstm_acc*100:.2f}%)')
print(f'ROC-AUC  : {lstm_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lstm, target_names=['Negative', 'Positive']))

fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(y_test, y_pred_lstm, 'BiLSTM - Confusion Matrix', ax)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

models_results = {
    'Model': ['Logistic Regression', 'Naive Bayes', 'BiLSTM'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred_lr)*100, 2),
        round(accuracy_score(y_test, y_pred_nb)*100, 2),
        round(accuracy_score(y_test, y_pred_lstm)*100, 2)
    ],
    'Precision': [
        round(precision_score(y_test, y_pred_lr)*100, 2),
        round(precision_score(y_test, y_pred_nb)*100, 2),
        round(precision_score(y_test, y_pred_lstm)*100, 2)
    ],
    'Recall': [
        round(recall_score(y_test, y_pred_lr)*100, 2),
        round(recall_score(y_test, y_pred_nb)*100, 2),
        round(recall_score(y_test, y_pred_lstm)*100, 2)
    ],
    'F1-Score': [
        round(f1_score(y_test, y_pred_lr)*100, 2),
        round(f1_score(y_test, y_pred_nb)*100, 2),
        round(f1_score(y_test, y_pred_lstm)*100, 2)
    ],
    'ROC-AUC': [
        round(lr_auc*100, 2),
        round(nb_auc*100, 2),
        round(lstm_auc*100, 2)
    ]
}

results_df = pd.DataFrame(models_results)
results_df = results_df.set_index('Model')
print('\n🏆 MODEL COMPARISON SUMMARY (%)')
print('=' * 65)
print(results_df.to_string())
print('\n⭐ Best Model by Accuracy:', results_df['Accuracy'].idxmax())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df.index))
width = 0.15
metrics = results_df.columns.tolist()
bar_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

for i, (metric, color) in enumerate(zip(metrics, bar_colors)):
    offset = (i - len(metrics)//2) * width
    bars = ax.bar(x + offset, results_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xlabel('Models', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, fontsize=11)
ax.legend(loc='lower right')
ax.set_ylim(60, 102)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for (name, y_prob, color) in [
    ('Logistic Regression', y_prob_lr, '#3498db'),
    ('Naive Bayes',         y_prob_nb, '#e74c3c'),
    ('BiLSTM',             y_prob_lstm, '#2ecc71')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = lr_model.coef_[0]

top_pos_idx = np.argsort(coefs)[-20:]
top_neg_idx = np.argsort(coefs)[:20]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Top Predictive Words (Logistic Regression Coefficients)', fontsize=14, fontweight='bold')

axes[0].barh(feature_names[top_pos_idx], coefs[top_pos_idx], color='#2ecc71')
axes[0].set_title('Top 20 POSITIVE Words', fontweight='bold')
axes[0].set_xlabel('Coefficient Value')

axes[1].barh(feature_names[top_neg_idx], coefs[top_neg_idx], color='#e74c3c')
axes[1].set_title('Top 20 NEGATIVE Words', fontweight='bold')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

In [ ]:
def predict_sentiment(text, model_type='lr'):
    clean = preprocess_text(text)

    if model_type == 'lr':
        vec   = tfidf.transform([clean])
        prob  = lr_model.predict_proba(vec)[0][1]
        label = 'POSITIVE ✅' if prob >= 0.5 else 'NEGATIVE ❌'

    elif model_type == 'nb':
        vec   = cv.transform([clean])
        prob  = nb_model.predict_proba(vec)[0][1]
        label = 'POSITIVE ✅' if prob >= 0.5 else 'NEGATIVE ❌'

    elif model_type == 'lstm':
        seq   = tokenizer.texts_to_sequences([clean])
        pad   = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
        prob  = lstm_model.predict(pad, verbose=0)[0][0]
        label = 'POSITIVE ✅' if prob >= 0.5 else 'NEGATIVE ❌'

    confidence = prob if prob >= 0.5 else 1 - prob
    print(f'Sentiment  : {label}')
    print(f'Confidence : {confidence*100:.1f}%')
    print(f'Positive Prob: {prob*100:.1f}%')
    return label, confidence

test_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout.",
    "Terrible waste of time. The story made no sense and the characters were boring and one-dimensional.",
    "It was okay, not the best film but had some good moments. The special effects were decent."
]

for i, review in enumerate(test_reviews, 1):
    print(f'\n--- Review {i} ---')
    print(f'Text: {review[:80]}...' if len(review)>80 else f'Text: {review}')
    predict_sentiment(review, model_type='lr')

In [ ]:
print('=' * 55)
print('🎬 INTERACTIVE SENTIMENT ANALYZER')
print('=' * 55)
user_review = input('Enter a movie review: ')

print('\n📊 Predictions from all 3 models:')
for name, mtype in [('Logistic Regression', 'lr'), ('Naive Bayes', 'nb'), ('BiLSTM', 'lstm')]:
    print(f'\n🔹 {name}:')
    predict_sentiment(user_review, model_type=mtype)